# 03 — PySpark Transformations

Builds cleaned datasets and a customer-level analytical view.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, concat_ws, to_date, year, month, count
spark=SparkSession.builder.appName('RetailTransformations').getOrCreate()
base='../Datasets/'
customers=spark.read.option('header',True).option('inferSchema',True).csv(base+'customers.csv')
orders=spark.read.option('header',True).option('inferSchema',True).csv(base+'orders_new.csv')


In [ ]:
customers_clean=(customers
    .dropDuplicates(['customer_id'])
    .withColumn('customer_fname', trim(col('customer_fname')))
    .withColumn('customer_lname', trim(col('customer_lname')))
    .withColumn('customer_name', concat_ws(' ', col('customer_fname'), col('customer_lname')))
)
orders_clean=(orders
    .dropDuplicates(['order_id'])
    .withColumn('order_date', col('order_date').cast('timestamp'))
    .withColumn('order_day', to_date('order_date'))
    .withColumn('order_year', year('order_date'))
    .withColumn('order_month', month('order_date'))
)


In [ ]:
customer_orders=(customers_clean.select('customer_id','customer_name','city','state','pincode')
    .join(orders_clean.groupBy('customer_id').agg(count('*').alias('order_count')), 'customer_id', 'left')
    .fillna({'order_count':0}))
customer_orders.orderBy(col('order_count').desc()).show(20, truncate=False)
